# exp_20260514_027_realmlp_shared009_min

Reference: `code_shared_009.txt`.

This notebook tests shared_009 RealMLP_TD as a potential additional NN branch.

Project adaptation: `Normalized_TyreLife` is dropped, and proxy-risk features (`TyreLife_Normalized`, `Norm_x_Hardness`, `Stint_x_Normalized`) are disabled by default.


In [1]:
# ============================================================
# PS S6E5 - exp_20260514_027_realmlp_shared009_min
#
# Reference:
#   code_shared_009.txt
#   RealMLP_TD via pytabkit, 6-fold CV, original rows appended
#
# Purpose:
#   Test shared_009 RealMLP as a possible additional NN branch.
#
# Important adaptation:
#   The source code creates TyreLife_Normalized / Norm_x_Hardness /
#   Stint_x_Normalized. In this project these are treated as
#   Normalized_TyreLife-proxy-risk features, so they are OFF by default.
#
#   Set CFG.USE_NORMALIZED_PROXY_FEATURES = True only if you intentionally
#   want to reproduce shared_009 as-is and accept that risk.
#
# Notes:
#   - PitStop is kept by default to match shared_009.
#   - Normalized_TyreLife from original is dropped.
#   - OOF/pred .npy are saved for blend.
# ============================================================

In [2]:
import os
import sys
import gc
import json
import random
import warnings
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)

In [3]:
# ============================================================
# Optional install / imports
# ============================================================

def ensure_pytabkit():
    try:
        import pytabkit  # noqa: F401
        return
    except Exception:
        print("pytabkit not found. Installing pytabkit...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pytabkit"])

ensure_pytabkit()

import torch
from pytabkit import RealMLP_TD_Classifier

pytabkit not found. Installing pytabkit...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 7.1 MB/s eta 0:00:00


In [4]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260514_027_realmlp_shared009_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    TEST_PATH = COMP_BASE / "test.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    ORIG_PATH = Path(
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv"
    )

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    TARGET = "PitNextLap"
    ID = "id"
    DANGER_COL = "Normalized_TyreLife"

    FOLDS = 6
    SEED = 42

    # shared_009 uses normal StratifiedKFold on target only.
    STRATIFY_BY_TARGET_YEAR = False

    # Keep PitStop by default to match shared_009.
    # If you want a safer no-PitStop branch, set True and rename the experiment.
    DROP_PITSTOP = False

    # Project guardrail:
    # Source shared_009 uses TyreLife_Normalized, Norm_x_Hardness, Stint_x_Normalized.
    # They are close to Normalized_TyreLife proxy, so default is False.
    USE_NORMALIZED_PROXY_FEATURES = False

    # Source shared_009 parameters.
    PARAMS = {
        "random_state": SEED,
        "verbosity": 2,
        "val_metric_name": "1-auc_ovr",

        # Architecture
        "n_ens": 32,
        "hidden_sizes": [512, 256, 128],
        "act": "silu",
        "embedding_size": 6,
        "max_one_hot_cat_size": 18,

        # Training
        "n_epochs": 10,
        "batch_size": 256,
        "lr": 0.03,
        "wd": 0.018,
        "sq_mom": 0.98,
        "lr_sched": "lin_cos_log_15",
        "first_layer_lr_factor": 0.25,

        # Regularization
        "p_drop": 0.05,
        "p_drop_sched": "expm4t",
        "ls_eps": 0.01,
        "ls_eps_sched": "sqrt_cos",

        # PBLD embeddings
        "plr_hidden_1": 16,
        "plr_hidden_2": 8,
        "plr_act_name": "gelu",
        "plr_lr_factor": 0.1151,
        "plr_sigma": 2.33,

        # Preprocessing
        "add_front_scale": False,
        "bias_init_mode": "neg-uniform-dynamic-2",
        "tfms": [
            "one_hot",
            "median_center",
            "robust_scale",
            "smooth_clip",
            "embedding",
            "l2_normalize",
        ],

        # Early stopping
        "use_early_stopping": False,
        "early_stopping_additive_patience": 10,
        "early_stopping_multiplicative_patience": 1,
    }


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [5]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def print_section(title: str) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def safe_auc(y_true, pred) -> float:
    return float(roc_auc_score(y_true, pred))


def per_year_auc(df: pd.DataFrame, y_true, pred) -> dict:
    out = {}
    years = sorted(pd.Series(df["Year"]).dropna().unique().tolist())
    y_arr = np.asarray(y_true)
    p_arr = np.asarray(pred)
    for yr in years:
        mask = (df["Year"].values == yr)
        if mask.sum() == 0 or len(np.unique(y_arr[mask])) < 2:
            out[str(int(yr))] = None
        else:
            out[str(int(yr))] = float(roc_auc_score(y_arr[mask], p_arr[mask]))
    return out


def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            c_min, c_max = out[col].min(), out[col].max()
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                out[col] = out[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                out[col] = out[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                out[col] = out[col].astype(np.int32)
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")
    return out


seed_everything(CFG.SEED)

print("PyTorch:", torch.__version__)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

PyTorch: 2.10.0+cu128
GPU    : Tesla T4


In [6]:
# ============================================================
# Load data
# ============================================================

print_section("Load Data")

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
sub = pd.read_csv(CFG.SUB_PATH)
orig = pd.read_csv(CFG.ORIG_PATH)

orig.drop(columns=[CFG.DANGER_COL], inplace=True, errors="ignore")

print("Train:", train.shape)
print("Test :", test.shape)
print("Orig :", orig.shape)
print("Target rate train:", train[CFG.TARGET].mean())
print("Target rate orig :", orig[CFG.TARGET].mean())

assert CFG.TARGET in train.columns
assert CFG.TARGET in orig.columns
assert CFG.ID in train.columns
assert CFG.ID in test.columns
assert CFG.DANGER_COL not in orig.columns


Load Data
Train: (439140, 16)
Test : (188165, 15)
Orig : (101371, 15)
Target rate train: 0.19898210137996994
Target rate orig : 0.25479673673930414


In [7]:
# ============================================================
# Feature Engineering
# ============================================================

COMPOUND_STINT_MEDIANS = {
    "SOFT": 14.0,
    "MEDIUM": 17.0,
    "HARD": 23.0,
    "INTERMEDIATE": 16.0,
    "WET": 14.0,
}

COMPOUND_HARDNESS = {
    "SOFT": 1,
    "MEDIUM": 2,
    "HARD": 3,
    "INTERMEDIATE": 1,
    "WET": 0,
}


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Core tyre features
    df["ExpectedStint"] = df["Compound"].map(COMPOUND_STINT_MEDIANS).fillna(17.0)

    if CFG.USE_NORMALIZED_PROXY_FEATURES:
        df["TyreLife_Normalized"] = df["TyreLife"] / df["ExpectedStint"]

    df["TyreLife_sq"] = df["TyreLife"] ** 2
    df["TyreLife_sqrt"] = np.sqrt(np.clip(df["TyreLife"], 0, None))
    df["TyreLife_log1p"] = np.log1p(np.clip(df["TyreLife"], 0, None))

    df["Compound_Hardness"] = df["Compound"].map(COMPOUND_HARDNESS).fillna(2).astype(int)
    df["TyreLife_x_Hardness"] = df["TyreLife"] * df["Compound_Hardness"]

    if CFG.USE_NORMALIZED_PROXY_FEATURES:
        df["Norm_x_Hardness"] = df["TyreLife_Normalized"] * df["Compound_Hardness"]

    # Tyre age flags
    df["Is_Fresh"] = (df["TyreLife"] <= 3).astype(np.int8)
    df["Is_Old"] = (df["TyreLife"] > 20).astype(np.int8)
    df["Is_VeryOld"] = (df["TyreLife"] > 40).astype(np.int8)
    df["Is_FirstStint"] = (df["Stint"] == 1).astype(np.int8)

    # Degradation
    df["DegRate"] = df["Cumulative_Degradation"] / (df["TyreLife"] + 1)

    # Race progress flags
    df["Early_Race"] = (df["RaceProgress"] < 0.25).astype(np.int8)
    df["Late_Race"] = (df["RaceProgress"] >= 0.75).astype(np.int8)
    df["VeryLate_Race"] = (df["RaceProgress"] >= 0.90).astype(np.int8)

    # Year flags
    df["Is_2023"] = (df["Year"] == 2023).astype(np.int8)
    df["Is_2025"] = (df["Year"] == 2025).astype(np.int8)

    # Ratio / interaction features
    df["TyreLife_per_Lap"] = df["TyreLife"] / df["LapNumber"].clip(lower=1)
    df["Lap_x_RaceProgress"] = df["LapNumber"] * df["RaceProgress"]
    df["Stint_x_TyreLife"] = df["Stint"] * df["TyreLife"]

    if CFG.USE_NORMALIZED_PROXY_FEATURES:
        df["Stint_x_Normalized"] = df["Stint"] * df["TyreLife_Normalized"]

    # LapTime signal
    df["LapDelta_sq"] = df["LapTime_Delta"] ** 2
    df["LapDelta_abs"] = df["LapTime_Delta"].abs()

    return df


print_section("Basic FE")

train = engineer_features(train)
test = engineer_features(test)
orig = engineer_features(orig)

y_orig = orig[CFG.TARGET].copy()
orig = orig.drop(columns=[CFG.TARGET])

y = train[CFG.TARGET].copy()
train_id = train[CFG.ID].copy()
test_id = test[CFG.ID].copy()

drop_cols_train = [CFG.ID, CFG.TARGET, "ExpectedStint"]
drop_cols_test = [CFG.ID, "ExpectedStint"]

if CFG.DROP_PITSTOP:
    drop_cols_train.append("PitStop")
    drop_cols_test.append("PitStop")

X = train.drop(columns=drop_cols_train, errors="ignore")
X_test = test.drop(columns=drop_cols_test, errors="ignore")
orig = orig.drop(columns=["ExpectedStint"], errors="ignore")

if CFG.DROP_PITSTOP:
    orig = orig.drop(columns=["PitStop"], errors="ignore")

orig = orig.reindex(columns=X.columns, fill_value=0)

print("X     :", X.shape)
print("X_test:", X_test.shape)
print("orig  :", orig.shape)

for c in ["Normalized_TyreLife", "TyreLife_Normalized", "Norm_x_Hardness", "Stint_x_Normalized"]:
    if c in X.columns:
        print(f"WARNING: proxy-risk feature enabled: {c}")


Basic FE
X     : (439140, 34)
X_test: (188165, 34)
orig  : (101371, 34)


In [8]:
# ============================================================
# NN-style categorical engineering
# ============================================================

category_map = {}

IMPORTANT_COMBOS = [
    ("Race", "Compound"),
    ("Race", "Year"),
    ("Driver", "Compound"),
]

NUM_COLS_BASE = [
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "RaceProgress",
    "Year",
]

# shared_009 includes PitStop as a numeric-to-cat source if present.
if "PitStop" in X.columns:
    NUM_COLS_BASE.append("PitStop")


def feature_engineering_nn(df: pd.DataFrame, fit: bool = False):
    df = df.copy()

    # Arithmetic interactions
    df["_TyreLife_div_LapNumber"] = (
        df["TyreLife"] / df["LapNumber"].clip(lower=1)
    ).astype("float32")

    df["_LapNumber_div_RaceProgress"] = (
        df["LapNumber"] / (df["RaceProgress"] + 1e-6)
    ).astype("float32")

    extra_num = ["_TyreLife_div_LapNumber", "_LapNumber_div_RaceProgress"]

    # Floor numericals -> categorical
    for col in NUM_COLS_BASE + extra_num:
        cat_name = f"{col}_cat_"
        if col in df.columns:
            floored = np.floor(pd.to_numeric(df[col], errors="coerce")).fillna(-999999)
            if fit:
                codes, uniques = pd.factorize(floored, sort=False)
                category_map[col] = uniques
            else:
                uniques = category_map.get(col, np.array([]))
                code_map = {cat: i for i, cat in enumerate(uniques)}
                codes = floored.map(code_map).fillna(-1).astype("int32")
            df[cat_name] = pd.Series(codes, index=df.index).astype(str)

    # Count encoding
    for col in ["Driver", "Compound", "Race", "Year"]:
        count_name = f"_{col}_count"
        if col in df.columns:
            if fit:
                count_map = df[col].astype(str).value_counts()
                category_map[count_name] = count_map
            else:
                count_map = category_map.get(count_name, pd.Series(dtype=int))
            df[count_name] = df[col].astype(str).map(count_map).fillna(0).astype("int32")

    # RaceProgress quantile binning
    bin_name = "RaceProgress_200_quantile_bin_"
    if "RaceProgress" in df.columns:
        if fit:
            kb = KBinsDiscretizer(
                n_bins=200,
                encode="ordinal",
                strategy="quantile",
                subsample=None,
            )
            binned = kb.fit_transform(df[["RaceProgress"]]).ravel().astype("int32")
            category_map[bin_name] = kb
        else:
            kb = category_map.get(bin_name)
            if kb is None:
                binned = np.zeros(len(df), dtype="int32")
            else:
                binned = kb.transform(df[["RaceProgress"]]).ravel().astype("int32")
        df[bin_name] = pd.Series(binned, index=df.index).astype(str)

    # Combo categorical features
    combo_names = []
    for cols in IMPORTANT_COMBOS:
        if not set(cols).issubset(df.columns):
            continue

        combo_name = "_".join(cols) + "_combo_"
        combo_names.append(combo_name)

        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + "_" + df[col].astype(str)

        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map.get(combo_name, np.array([]))
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype("int32")

        df[combo_name] = pd.Series(codes, index=df.index).astype(str)

    return df, combo_names


print_section("NN FE")

X, combo_names = feature_engineering_nn(X, fit=True)
X_test, _ = feature_engineering_nn(X_test, fit=False)
orig, _ = feature_engineering_nn(orig, fit=False)

orig = orig.reindex(columns=X.columns, fill_value=0)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

X = reduce_mem_usage(X)
X_test = reduce_mem_usage(X_test)
orig = reduce_mem_usage(orig)

print("X after NN FE     :", X.shape)
print("X_test after NN FE:", X_test.shape)
print("orig after NN FE  :", orig.shape)
print("combo features    :", combo_names)


NN FE
X after NN FE     : (439140, 54)
X_test after NN FE: (188165, 54)
orig after NN FE  : (101371, 54)
combo features    : ['Race_Compound_combo_', 'Race_Year_combo_', 'Driver_Compound_combo_']


In [9]:
# ============================================================
# CV Training
# ============================================================

print_section("CV Training")

if CFG.STRATIFY_BY_TARGET_YEAR:
    strat_key = y.astype(str) + "__" + train["Year"].astype(str)
else:
    strat_key = y

skf = StratifiedKFold(
    n_splits=CFG.FOLDS,
    shuffle=True,
    random_state=CFG.SEED,
)

oof_preds = np.zeros(len(X), dtype=np.float32)
test_preds = np.zeros(len(X_test), dtype=np.float32)

fold_scores = []
fold_target_means = []
te_names_final = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, strat_key), 1):
    print("\n" + "=" * 80)
    print(f"FOLD {fold}/{CFG.FOLDS}")
    print("=" * 80)

    X_tr = X.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_val = y.iloc[val_idx].copy()
    X_tst = X_test.copy()

    # Add original dataset to training fold only.
    X_tr = pd.concat(
        [X_tr.reset_index(drop=True), orig.reset_index(drop=True)],
        axis=0,
        ignore_index=True,
    )
    y_tr = pd.concat(
        [y_tr.reset_index(drop=True), y_orig.reset_index(drop=True)],
        axis=0,
        ignore_index=True,
    )

    # Target encoding on combo features inside fold.
    te_cols = combo_names
    if len(te_cols) > 0:
        te = TargetEncoder(
            cv=CFG.FOLDS,
            smooth="auto",
            shuffle=True,
            random_state=CFG.SEED,
        )

        tr_enc = te.fit_transform(X_tr[te_cols], y_tr)
        val_enc = te.transform(X_val[te_cols])
        tst_enc = te.transform(X_tst[te_cols])

        te_names = [f"_{c}_TE" for c in te_cols]
        te_names_final = te_names

        X_tr[te_names] = tr_enc
        X_val[te_names] = val_enc
        X_tst[te_names] = tst_enc

    if fold == 1:
        print("Total features:", X_tr.shape[1])
        print("Train rows    :", len(X_tr), "(synthetic fold-train + original)")
        pd.Series(X_tr.columns).to_csv(CFG.OUTDIR / "feature_columns.csv", index=False, header=False)

    model = RealMLP_TD_Classifier(**CFG.PARAMS)
    model.fit(X_tr, y_tr, X_val, y_val)

    val_preds = model.predict_proba(X_val)[:, 1].astype(np.float32)
    fold_test_preds = model.predict_proba(X_tst)[:, 1].astype(np.float32)

    oof_preds[val_idx] = val_preds
    test_preds += fold_test_preds / CFG.FOLDS

    fold_auc = safe_auc(y_val, val_preds)
    fold_scores.append(fold_auc)
    fold_target_means.append(
        {
            "fold": fold,
            "train_target_mean_with_orig": float(np.mean(y_tr)),
            "valid_target_mean": float(np.mean(y_val)),
            "fold_auc": fold_auc,
        }
    )

    print(f"Fold {fold} AUC: {fold_auc:.9f}")

    del X_tr, X_val, X_tst, model, val_preds, fold_test_preds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


CV Training

FOLD 1/6
Total features: 57
Train rows    : 467321 (synthetic fold-train + original)
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'TyreLife_sq', 'TyreLife_sqrt', 'TyreLife_log1p', 'Compound_Hardness', 'TyreLife_x_Hardness', 'Is_Fresh', 'Is_Old', 'Is_VeryOld', 'Is_FirstStint', 'DegRate', 'Early_Race', 'Late_Race', 'VeryLate_Race', 'Is_2023', 'Is_2025', 'TyreLife_per_Lap', 'Lap_x_RaceProgress', 'Stint_x_TyreLife', 'LapDelta_sq', 'LapDelta_abs', '_TyreLife_div_LapNumber', '_LapNumber_div_RaceProgress', '_Driver_count', '_Compound_count', '_Race_count', '_Year_count', '_Race_Compound_combo__TE', '_Race_Year_combo__TE', '_Driver_Compound_combo__TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'RaceProgress_cat_', 'Year_cat_',

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/10: val 1-auc_ovr = 0.058020
Epoch 2/10: val 1-auc_ovr = 0.049787
Epoch 3/10: val 1-auc_ovr = 0.047293
Epoch 4/10: val 1-auc_ovr = 0.045521
Epoch 5/10: val 1-auc_ovr = 0.045259
Epoch 6/10: val 1-auc_ovr = 0.045658
Epoch 7/10: val 1-auc_ovr = 0.045801
Epoch 8/10: val 1-auc_ovr = 0.045312
Epoch 9/10: val 1-auc_ovr = 0.045337
Epoch 10/10: val 1-auc_ovr = 0.046215


`Trainer.fit` stopped: `max_epochs=10` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 1 AUC: 0.954740748

FOLD 2/6
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'TyreLife_sq', 'TyreLife_sqrt', 'TyreLife_log1p', 'Compound_Hardness', 'TyreLife_x_Hardness', 'Is_Fresh', 'Is_Old', 'Is_VeryOld', 'Is_FirstStint', 'DegRate', 'Early_Race', 'Late_Race', 'VeryLate_Race', 'Is_2023', 'Is_2025', 'TyreLife_per_Lap', 'Lap_x_RaceProgress', 'Stint_x_TyreLife', 'LapDelta_sq', 'LapDelta_abs', '_TyreLife_div_LapNumber', '_LapNumber_div_RaceProgress', '_Driver_count', '_Compound_count', '_Race_count', '_Year_count', '_Race_Compound_combo__TE', '_Race_Year_combo__TE', '_Driver_Compound_combo__TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'RaceProgress_cat_', 'Year_cat_', 'PitStop_cat_', '_TyreLife_div_LapNumber_cat_', '_LapNumber_div_

...

Fold 5 AUC: 0.952400752

FOLD 6/6
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'TyreLife_sq', 'TyreLife_sqrt', 'TyreLife_log1p', 'Compound_Hardness', 'TyreLife_x_Hardness', 'Is_Fresh', 'Is_Old', 'Is_VeryOld', 'Is_FirstStint', 'DegRate', 'Early_Race', 'Late_Race', 'VeryLate_Race', 'Is_2023', 'Is_2025', 'TyreLife_per_Lap', 'Lap_x_RaceProgress', 'Stint_x_TyreLife', 'LapDelta_sq', 'LapDelta_abs', '_TyreLife_div_LapNumber', '_LapNumber_div_RaceProgress', '_Driver_count', '_Compound_count', '_Race_count', '_Year_count', '_Race_Compound_combo__TE', '_Race_Year_combo__TE', '_Driver_Compound_combo__TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'RaceProgress_cat_', 'Year_cat_', 'PitStop_cat_', '_TyreLife_div_LapNumber_cat_', '_LapNumber_div_

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/10: val 1-auc_ovr = 0.058540
Epoch 2/10: val 1-auc_ovr = 0.050132
Epoch 3/10: val 1-auc_ovr = 0.047878
Epoch 4/10: val 1-auc_ovr = 0.046036
Epoch 5/10: val 1-auc_ovr = 0.045882
Epoch 6/10: val 1-auc_ovr = 0.046186
Epoch 7/10: val 1-auc_ovr = 0.046283
Epoch 8/10: val 1-auc_ovr = 0.045867
Epoch 9/10: val 1-auc_ovr = 0.046143
Epoch 10/10: val 1-auc_ovr = 0.047197


`Trainer.fit` stopped: `max_epochs=10` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 6 AUC: 0.954133322


In [10]:
# ============================================================
# Results
# ============================================================

print_section("Results")

oof_auc = safe_auc(y, oof_preds)
year_auc = per_year_auc(train, y.values, oof_preds)

print(f"OOF ROC-AUC: {oof_auc:.9f}")
print("fold_scores:", fold_scores)
print("per_year_auc:", year_auc)

# sanity
assert len(oof_preds) == len(train)
assert len(test_preds) == len(test)
assert np.isfinite(oof_preds).all()
assert np.isfinite(test_preds).all()
assert CFG.DANGER_COL not in X.columns

if not CFG.USE_NORMALIZED_PROXY_FEATURES:
    for c in ["TyreLife_Normalized", "Norm_x_Hardness", "Stint_x_Normalized"]:
        assert c not in X.columns, f"{c} should not be used in safe 027"


Results
OOF ROC-AUC: 0.953140078
fold_scores: [0.9547407477335534, 0.9519910222920218, 0.9523272927612594, 0.953439464908098, 0.9524007520014982, 0.9541333222601319]
per_year_auc: {'2022': 0.9181797611120248, '2023': 0.9442708835475394, '2024': 0.9331191191776498, '2025': 0.9324853799673106}


In [11]:
# ============================================================
# Save artifacts
# ============================================================

print_section("Save artifacts")

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof_preds)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", test_preds)

oof_df = pd.DataFrame({
    CFG.ID: train_id.values,
    CFG.TARGET: oof_preds,
})
oof_df.to_csv(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv", index=False)

target_col = [c for c in sub.columns if c != CFG.ID][0]
sub_out = sub.copy()
sub_out[target_col] = np.clip(test_preds, 1e-6, 1 - 1e-6)

submission_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub_out.to_csv(submission_path, index=False)

feature_flags = pd.DataFrame({
    "feature": X.columns.tolist() + te_names_final,
})
feature_flags["is_combo"] = feature_flags["feature"].isin(combo_names)
feature_flags["is_te"] = feature_flags["feature"].isin(te_names_final)
feature_flags["is_cat_floor"] = feature_flags["feature"].str.endswith("_cat_")
feature_flags["is_count"] = feature_flags["feature"].str.endswith("_count")
feature_flags["is_proxy_risk"] = feature_flags["feature"].isin([
    "TyreLife_Normalized",
    "Norm_x_Hardness",
    "Stint_x_Normalized",
])
feature_flags.to_csv(CFG.OUTDIR / "feature_columns_detail.csv", index=False)

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "created_at": "2026-05-14",
    },
    "source": {
        "reference_code": "code_shared_009.txt",
        "model": "RealMLP_TD_Classifier via pytabkit",
        "adaptation": [
            "OOF/pred npy saving added.",
            "memo_draft.yml added.",
            "Normalized_TyreLife is dropped from original data.",
            "TyreLife_Normalized / Norm_x_Hardness / Stint_x_Normalized are disabled by default.",
            "PitStop is kept by default to match shared_009.",
        ],
    },
    "config": {
        "folds": CFG.FOLDS,
        "seed": CFG.SEED,
        "stratify_by_target_year": CFG.STRATIFY_BY_TARGET_YEAR,
        "drop_pitstop": CFG.DROP_PITSTOP,
        "use_normalized_proxy_features": CFG.USE_NORMALIZED_PROXY_FEATURES,
        "params": CFG.PARAMS,
    },
    "data": {
        "train_shape": list(train.shape),
        "test_shape": list(test.shape),
        "orig_shape_after_drop": list(orig.shape),
        "target_rate_train": float(train[CFG.TARGET].mean()),
        "target_rate_orig": float(y_orig.mean()),
    },
    "features": {
        "n_features_before_fold_te": int(X.shape[1]),
        "combo_names": combo_names,
        "te_names": te_names_final,
        "danger_col_used": False,
        "proxy_risk_features_used": bool(CFG.USE_NORMALIZED_PROXY_FEATURES),
        "pitstop_used": bool("PitStop" in X.columns),
    },
    "result": {
        "cv_auc": float(oof_auc),
        "fold_scores": [float(x) for x in fold_scores],
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
        "per_year_auc": year_auc,
        "fold_target_means": fold_target_means,
        "public_lb": None,
    },
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "oof_npy": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred_npy": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "oof_csv": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv"),
        "submission": str(submission_path),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "feature_columns_detail": str(CFG.OUTDIR / "feature_columns_detail.csv"),
        "cv_result": str(CFG.OUTDIR / "cv_result.json"),
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)


Save artifacts


In [12]:
memo_yml = f"""experiment:
  id: exp_20260514_027_realmlp_shared009_min
  title: RealMLP shared009 safe adaptation
  competition: PS S6E5 Predicting F1 Pit Stops
  metric: AUC
  status: completed

objective:
  summary: >
    shared_009 RealMLP_TD notebookを参照し、既存018/022/023 RealMLP branchとは異なるNN素材になるか確認する。
    OOF/predをnpy保存し、026-only compressed stackへ追加してblend寄与を見る。
  intended_role: new_realmlp_branch_candidate

source:
  reference_code: code_shared_009.txt
  original_shared009_reported:
    cv_auc: 0.953149
    public_lb: 0.95253
  notes:
    - RealMLP_TD via pytabkit
    - 6-fold CV
    - original rows appended to each fold train
    - combo target encoding inside fold
    - floored numericals treated as categorical variables

project_adaptation:
  normalized_tyre_life:
    original_column_dropped: true
    proxy_features_enabled: false
    disabled_features:
      - TyreLife_Normalized
      - Norm_x_Hardness
      - Stint_x_Normalized
    reason: >
      These features are close to Normalized_TyreLife proxy construction,
      so they are disabled by default in this project.
  pitstop:
    used: {str(not CFG.DROP_PITSTOP).lower()}
    note: >
      PitStop is kept by default to match shared_009.
      If this branch is useful but suspicious, create a no-PitStop ablation later.

data:
  competition_train: /kaggle/input/competitions/playground-series-s6e5/train.csv
  competition_test: /kaggle/input/competitions/playground-series-s6e5/test.csv
  original: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv
  target: PitNextLap

features:
  base:
    - TyreLife_sq
    - TyreLife_sqrt
    - TyreLife_log1p
    - Compound_Hardness
    - TyreLife_x_Hardness
    - Is_Fresh
    - Is_Old
    - Is_VeryOld
    - Is_FirstStint
    - DegRate
    - Early_Race
    - Late_Race
    - VeryLate_Race
    - Is_2023
    - Is_2025
    - TyreLife_per_Lap
    - Lap_x_RaceProgress
    - Stint_x_TyreLife
    - LapDelta_sq
    - LapDelta_abs
  nn_view:
    - floored_numeric_categories
    - count_encoding_for_Driver_Compound_Race_Year
    - RaceProgress_200_quantile_bin
    - Race_Compound_combo
    - Race_Year_combo
    - Driver_Compound_combo
    - fold_safe_TE_for_combo_features

model:
  family: RealMLP
  library: pytabkit
  estimator: RealMLP_TD_Classifier
  folds: 6
  seed: 42
  params:
    n_ens: 32
    hidden_sizes:
      - 512
      - 256
      - 128
    n_epochs: 10
    batch_size: 256
    lr: 0.03
    wd: 0.018
    ls_eps: 0.01
    val_metric_name: 1-auc_ovr

result:
  cv_auc: {float(oof_auc)}
  public_lb: null
  fold_scores: {[float(x) for x in fold_scores]}
  fold_mean: {float(np.mean(fold_scores))}
  fold_std: {float(np.std(fold_scores))}  
  per_year_auc: {year_auc}
  compared_to_existing_realmlp:
    "007":
      cv_auc: 0.9532701289015783
      public_lb: 0.95273
    "018":
      cv_auc: 0.953475993350039
      public_lb: 0.95316
    "022":
      cv_auc: 0.9534314709948207
      public_lb: 0.95301
    "023":
      cv_auc: 0.9534462611732204
      public_lb: 0.95304

artifacts:
  notebook: exp_20260514_027_realmlp_shared009_min.ipynb
  oof: oof_exp_20260514_027_realmlp_shared009_min.npy
  pred: pred_exp_20260514_027_realmlp_shared009_min.npy
  submission: submission_exp_20260514_027_realmlp_shared009_min.csv
  cv_result: cv_result.json
  feature_columns: feature_columns.csv
  feature_columns_detail: feature_columns_detail.csv

blend_policy:
  benchmark_code: code_010_add_slsqp_drop020.txt
  add_to_stack:
    - "007"
    - "014"
    - "015"
    - "016"
    - "017"
    - "018"
    - "021"
    - "022"
    - "023"
    - "026"
    - "027"
  decision_focus:
    - single_cv_vs_018_022_023
    - corr_vs_018_022_023
    - nm_hc_slsqp_weight
    - public_lb_if_submitted
  keep_rule: >
    Keep if CV is near existing RealMLP branch and correlation is meaningfully lower,
    or if blend weight remains at least around 0.02.
  drop_rule: >
    Drop if CV is weak, correlation with 018/022/023 is very high,
    and blend weight is near zero.

judgement:
  status: pending
  summary: >
    CV/LB/correlation/blend weight確認後に keep / hold / drop を判断する。
"""
with open(CFG.OUTDIR / "memo_draft.yml", "w", encoding="utf-8") as f:
    f.write(memo_yml)

print("Saved to:", CFG.OUTDIR)
print("Submission:", submission_path)
display(sub_out.head())

Saved to: /kaggle/working/exp_20260514_027_realmlp_shared009_min
Submission: /kaggle/working/exp_20260514_027_realmlp_shared009_min/submission_exp_20260514_027_realmlp_shared009_min.csv


,id,PitNextLap
0,439140,0.007018
1,439141,0.018456
2,439142,0.005981
3,439143,0.281110
4,439144,0.775732
